# Libraries

In [ ]:
import warnings, torch
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearnex import patch_sklearn
patch_sklearn()
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.pipeline import FeatureUnion
import gc

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


# Training ML models

In [3]:
# for restart
train = pd.read_csv('./data/train_lemmatized.csv')
test = pd.read_csv('./data/test_lemmatized.csv')
val = pd.read_csv("./data/val_lemmatized.csv")
train = train.dropna(subset=['message_lemmatized', 'label'])
test = test.dropna(subset=['message_lemmatized', 'label'])
misinfo, propaganda, opposition = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
misinfo = train[train.label == 0]
propaganda = train[train.label == 1]
opposition = train[train.label == 2]

In [4]:
df = pd.concat([
    misinfo.assign(label='misinformation'),
    propaganda.assign(label='propaganda'),
    opposition.assign(label='opposition')
], ignore_index=True)
df.head(5)

,Unnamed: 0,date,message,views,forwards,label,message_length,message_lemmatized
0,1,2022-02-18 15:39:01+00:00,очень тревожно на донбассе очень,24552.0,6.0,misinformation,5,очень тревожный донбасс очень
1,2,2022-02-24 18:37:53+00:00,подписчики сообщают что все цветочные магазины...,43939.0,160.0,misinformation,13,подписчик сообщать цветочный магазин киев пуст...
2,3,2022-02-10 17:34:25+00:00,ох как прав леонид эдуардович уважаемый с этим...,2514.0,1.0,misinformation,55,ох право леонид эдуардович уважаемый это свой ...
3,4,2022-02-21 11:28:17+00:00,политсоветники нормандской четверки смогут вст...,23272.0,21.0,misinformation,17,политсоветник нормандский четвёрка смочь встре...
4,5,2022-01-24 13:07:06+00:00,киевская экономика пожухла таллин наконецто уз...,18849.0,12.0,misinformation,12,киевский экономика пожухлый таллин наконецтый ...


In [5]:
X = df['message_lemmatized']
y = df['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [6]:
X_test

3157     встреча путин женева байден подтвердить безаль...
9220     27 январь смм обсе зафиксировать 266 нарушение...
18342    лукашенко заявить считать бог белорус фраза ру...
1542     огромный плакат изображение президент россия в...
4258     президент россия владимир путин наблюдать обос...
                               ...                        
25564    кремль успеть ознакомиться публикация команда ...
26214    путин обвинить украина блицкриг донбасс 201420...
22112    кличко погорячиться видимо сказаться нервный п...
13023    оперативный группа новосибирский область прове...
16309    слух план ввести военный положение ряд регион ...
Name: message_lemmatized, Length: 6128, dtype: object

In [7]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

models = {
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=200, 
                                            random_state=42,
                                            n_jobs=-1),
    "SVM": SVC(kernel='linear', 
               probability=True, 
               random_state=42),
    "LightGBM": lgb.LGBMClassifier(n_estimators=500, 
                                   random_state=42,        
                                   n_jobs=-1)
}

for name, model in models.items():
    print(f"\n--- {name} ---")
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


--- Naive Bayes ---
Classification Report:
                precision    recall  f1-score   support

misinformation       0.68      0.77      0.72      2092
    opposition       0.77      0.64      0.70      1997
    propaganda       0.60      0.62      0.61      2039

      accuracy                           0.68      6128
     macro avg       0.68      0.68      0.68      6128
  weighted avg       0.68      0.68      0.68      6128

Confusion Matrix:
[[1601  108  383]
 [ 266 1279  452]
 [ 491  278 1270]]

--- Random Forest ---
Classification Report:
                precision    recall  f1-score   support

misinformation       0.68      0.79      0.73      2092
    opposition       0.77      0.66      0.71      1997
    propaganda       0.63      0.61      0.62      2039

      accuracy                           0.69      6128
     macro avg       0.69      0.69      0.69      6128
  weighted avg       0.69      0.69      0.69      6128

Confusion Matrix:
[[1644  114  334]
 [ 276 1318

# Embeddings 

In [8]:
vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=10000, lowercase=True)

X = vectorizer.fit_transform(train['message_lemmatized'].dropna())
feature_names = vectorizer.get_feature_names_out()

In [9]:
top_ngrams_by_class = {}
labels = [0, 1, 2]

for class_label in set(labels):
    idx = [i for i, lbl in enumerate(labels) if lbl == class_label]
    class_matrix = X[idx]
    scores = class_matrix.mean(axis=0).A1
    top_n_idx = scores.argsort()[-20:][::-1]
    top_ngrams = [(feature_names[i], scores[i]) for i in top_n_idx]
    top_ngrams_by_class[class_label] = top_ngrams

In [10]:
list(vectorizer.vocabulary_.items())[:10]

[('зеленский', np.int64(2822)),
 ('сообщить', np.int64(8246)),
 ('настаивать', np.int64(4526)),
 ('введение', np.int64(852)),
 ('санкция', np.int64(7669)),
 ('отношение', np.int64(5432)),
 ('россия', np.int64(7435)),
 ('введение санкция', np.int64(853)),
 ('санкция отношение', np.int64(7677)),
 ('отношение россия', np.int64(5435))]

In [11]:
for label, ngrams in top_ngrams_by_class.items():
    print(f"\nTop n-grams for class {label}:")
    for phrase, score in ngrams:
        if score > 0:
            print(f"{phrase}: {score:.4f}")


Top n-grams for class 0:
санкция отношение россия: 0.4178
введение санкция: 0.3852
настаивать: 0.3693
санкция отношение: 0.3566
отношение россия: 0.3359
введение: 0.3071
отношение: 0.2355
зеленский: 0.2329
санкция: 0.2150
сообщить: 0.1865
россия: 0.1250

Top n-grams for class 1:
очень: 0.7351
тревожный: 0.6139
донбасс: 0.2876

Top n-grams for class 2:
пустой: 0.5044
подписчик: 0.4356
магазин: 0.4005
ждать: 0.3501
сообщать: 0.2934
город: 0.2825
киев: 0.2702
российский: 0.1847


# Testing model with embedding

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=10000, lowercase=True)
train_clean = train.dropna(subset=['message_lemmatized', 'label'])
X_train = vectorizer.fit_transform(train_clean['message_lemmatized'])
y_train = train_clean['label'].values

test_clean = test.dropna(subset=['message_lemmatized', 'label'])
X_test = vectorizer.transform(test_clean['message_lemmatized'])
y_test = test_clean['label'].values

logreg = LogisticRegression(
    multi_class='multinomial',
    solver='saga',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)
y_pred_proba = logreg.predict_proba(X_test)

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[f'Class {i}' for i in range(len(np.unique(y_train)))]))

print("\n" + "-"*50)
print("TOP FEATURES PER CLASS")

feature_names = vectorizer.get_feature_names_out()
for class_idx in range(logreg.coef_.shape[0]):
    mapping = {0: 'misinformation',
               1: 'propaganda',
               2: 'opposition'}
    print(f"\nTop 20 features for Class {mapping[class_idx]}:")
    top_indices = logreg.coef_[class_idx].argsort()[-20:][::-1]
    for idx in top_indices:
        print(f"  {feature_names[idx]}: {logreg.coef_[class_idx][idx]:.4f}")

tfidf_results = pd.DataFrame([{
    'method': 'TF-IDF (1-3 grams)',
    'classifier': 'LR',
    'accuracy': accuracy,
    'f1_macro': report['macro avg']['f1-score'],
    'precision_macro': report['macro avg']['precision'],
    'recall_macro': report['macro avg']['recall'],
}])

tfidf_results.to_csv('tfidf_results.csv', index=False)

Confusion Matrix:
[[2685  587  204]
 [ 687 2241  468]
 [ 350  611 2384]]

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.72      0.77      0.75      3476
     Class 1       0.65      0.66      0.66      3396
     Class 2       0.78      0.71      0.74      3345

    accuracy                           0.72     10217
   macro avg       0.72      0.72      0.72     10217
weighted avg       0.72      0.72      0.72     10217


--------------------------------------------------
TOP FEATURES PER CLASS

Top 20 features for Class misinformation:
  донбасс: 5.6510
  решать: 3.2192
  донбасс решать: 3.0394
  канал донбасс: 2.9877
  канал донбасс решать: 2.9868
  лнр: 2.8717
  днр: 2.6548
  донецк: 2.3871
  гаспарян: 2.3609
  всу: 2.1731
  тихановский: 2.1357
  изолента: 2.1271
  донецкий: 2.1236
  украинский: 2.1069
  минобороны рф: 2.1020
  мразь: 2.0759
  луганский: 1.9976
  добрый: 1.9445
  посёлок: 1.8927
  луганск: 1.8772

Top 20 features 

In [13]:
train.dropna(inplace=True)
test.dropna(inplace=True)
val.dropna(inplace=True)

In [14]:
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

In [15]:
train_clean = train.dropna(subset=['message_lemmatized', 'label'])
X_train_embed = model.encode(train['message_lemmatized'].tolist(), batch_size=64, show_progress_bar=True)
y_train = train_clean['label'].values
val_clean = val.dropna(subset=['message_lemmatized', 'label'])
X_test_embed = model.encode(val_clean['message_lemmatized'].astype(str).tolist(), batch_size=64, show_progress_bar=True)
y_test = val_clean['label'].values

Batches:   0%|          | 0/479 [00:00<?, ?it/s]

Batches:   0%|          | 0/160 [00:00<?, ?it/s]

In [16]:
clf = LogisticRegression(
        max_iter=1000,
        random_state=42,
        n_jobs=-1,
    )
clf.fit(X_train_embed, y_train)

y_pred = clf.predict(X_test_embed)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=[f'Class {i}' for i in np.unique(y_train)]))

Accuracy: 0.5364
Confusion Matrix:
 [[2229  832  441]
 [1057 1686  641]
 [ 783  986 1570]]
Classification Report:
               precision    recall  f1-score   support

     Class 0       0.55      0.64      0.59      3502
     Class 1       0.48      0.50      0.49      3384
     Class 2       0.59      0.47      0.52      3339

    accuracy                           0.54     10225
   macro avg       0.54      0.53      0.53     10225
weighted avg       0.54      0.54      0.53     10225



In [ ]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

EMBEDDING_MODELS = {
    'all-minilm': 'all-MiniLM-L6-v2',
    'paraphrase-multilingual-minilm': 'paraphrase-multilingual-MiniLM-L12-v2',
    'paraphrase-xlm-r': 'paraphrase-xlm-r-multilingual-v1',
    'bert-base-uncased': 'bert-base-uncased',
    'bert-multilingual': 'bert-base-multilingual-cased',
    'rubert': 'DeepPavlov/rubert-base-cased',
}

results_all = []

In [ ]:
print("="*70)

for model_key, model_path in EMBEDDING_MODELS.items():
    print(f"\n[{model_key}] Encoding...")
    
    try:
        clear_memory()
        
        model = SentenceTransformer(model_path, device=device)
        
        X_train = model.encode(
            train['message_lemmatized'].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )
        
        X_test = model.encode(
            test['message_lemmatized'].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )
        
        del model
        clear_memory()
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        y_train = train['label'].values
        y_test = test['label'].values
        
        print(f"  [LogisticRegression] Training...")
        lr_clf = LogisticRegression(max_iter=1000,
                                    random_state=42,
                                    solver='saga',
                                    multi_class='multinomial',
                                    n_jobs=-1)
        lr_clf.fit(X_train_scaled, y_train)
        
        lr_pred = lr_clf.predict(X_test_scaled)
        lr_acc = accuracy_score(y_test, lr_pred)
        lr_f1 = f1_score(y_test, lr_pred, average='macro')
        lr_report = classification_report(y_test, lr_pred, output_dict=True, zero_division=0)
        
        print(f"    Accuracy: {lr_acc:.4f} | F1: {lr_f1:.4f}")
        
        results_all.append({
            'model': model_key,
            'classifier': 'LR',
            'accuracy': lr_acc,
            'f1_macro': lr_f1,
            'precision_macro': lr_report['macro avg']['precision'],
            'recall_macro': lr_report['macro avg']['recall'],
        })
        
        print(f"  [RandomForest] Training...")
        rf_clf = RandomForestClassifier(
            n_estimators=200,
            max_depth=20,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        )
        rf_clf.fit(X_train_scaled, y_train)
        
        rf_pred = rf_clf.predict(X_test_scaled)
        rf_acc = accuracy_score(y_test, rf_pred)
        rf_f1 = f1_score(y_test, rf_pred, average='macro')
        rf_report = classification_report(y_test, rf_pred, output_dict=True, zero_division=0)
        
        print(f"    Accuracy: {rf_acc:.4f} | F1: {rf_f1:.4f}")
        
        results_all.append({
            'model': model_key,
            'classifier': 'RF',
            'accuracy': rf_acc,
            'f1_macro': rf_f1,
            'precision_macro': rf_report['macro avg']['precision'],
            'recall_macro': rf_report['macro avg']['recall'],
        })
        
        del X_train, X_test, X_train_scaled, X_test_scaled
        clear_memory()
        
    except Exception as e:
        print(f"  ERROR: {e}")
        clear_memory()
        continue

6 MODELS x 2 CLASSIFIERS WITH GRIDSEARCH

[all-minilm] Encoding...


Batches:   0%|          | 0/479 [00:00<?, ?it/s]

Batches:   0%|          | 0/160 [00:00<?, ?it/s]

  [LogisticRegression] Training...
    Accuracy: 0.5375 | F1: 0.5362
  [RandomForest] Training...
    Accuracy: 0.5364 | F1: 0.5329

[paraphrase-multilingual-minilm] Encoding...


Batches:   0%|          | 0/479 [00:00<?, ?it/s]

...

  [LogisticRegression] Training...
    Accuracy: 0.6375 | F1: 0.6361
  [RandomForest] Training...
    Accuracy: 0.6058 | F1: 0.6038

[rubert] Encoding...


No sentence-transformers model found with name DeepPavlov/rubert-base-cased. Creating a new one with mean pooling.
Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model

Batches:   0%|          | 0/479 [00:00<?, ?it/s]

Batches:   0%|          | 0/160 [00:00<?, ?it/s]

  [LogisticRegression] Training...
    Accuracy: 0.6462 | F1: 0.6451
  [RandomForest] Training...
    Accuracy: 0.6221 | F1: 0.6211


In [ ]:
results_df = pd.DataFrame(results_all)
results_df.to_csv('simple_results.csv', index=False)

print("\n" + "="*70)
print("All results")
print("="*70)
print(results_df.to_string(index=False))

print("\n" + "="*70)
print("Best Model")
print("="*70)
best = results_df.loc[results_df['f1_macro'].idxmax()]
print(f"Model: {best['model']}")
print(f"Classifier: {best['classifier']}")
print(f"Accuracy: {best['accuracy']:.4f}")
print(f"F1: {best['f1_macro']:.4f}")


ALL 12 RESULTS
                         model         classifier  accuracy  f1_macro  precision_macro  recall_macro
                    all-minilm LogisticRegression  0.539101  0.537382         0.540785      0.538180
                    all-minilm LogisticRegression  0.537535  0.536204         0.538116      0.536756
                    all-minilm       RandomForest  0.536361  0.532863         0.577251      0.534730
paraphrase-multilingual-minilm LogisticRegression  0.628267  0.626987         0.627180      0.627717
paraphrase-multilingual-minilm       RandomForest  0.614955  0.613060         0.619211      0.613940
              paraphrase-xlm-r LogisticRegression  0.636586  0.635462         0.635696      0.636024
              paraphrase-xlm-r       RandomForest  0.617011  0.615990         0.622030      0.616042
             bert-base-uncased LogisticRegression  0.603112  0.602155         0.602548      0.602566
             bert-base-uncased       RandomForest  0.544680  0.541738      

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=30000,
    lowercase=True
)
X_train = vectorizer.fit_transform(train['message_lemmatized'])
X_test = vectorizer.transform(test['message_lemmatized'])

y_train = train['label'].values
y_test = test['label'].values

clf = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1,
    solver='saga',
    multi_class='multinomial'
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("\n" + "="*70)
print("RESULTS")
print("="*70)
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1 (macro): {f1:.4f}")
print(f"Precision: {report['macro avg']['precision']:.4f}")
print(f"Recall:    {report['macro avg']['recall']:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(cm)

results = {
    'method': 'TF-IDF (unigram)',
    'classifier': 'LR' ,
    'accuracy': accuracy,
    'f1_macro': f1,
    'precision_macro': report['macro avg']['precision'],
    'recall_macro': report['macro avg']['recall'],
}

results_df = pd.DataFrame([results])
results_df.to_csv('unigrams_results.csv', index=False)


RESULTS
Accuracy:  0.7262
F1 (macro): 0.7263
Precision: 0.7293
Recall:    0.7257

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.79      0.76      3476
           1       0.66      0.67      0.67      3396
           2       0.80      0.72      0.75      3345

    accuracy                           0.73     10217
   macro avg       0.73      0.73      0.73     10217
weighted avg       0.73      0.73      0.73     10217


Confusion Matrix:
[[2745  554  177]
 [ 683 2283  430]
 [ 331  622 2392]]


In [ ]:
import joblib

joblib.dump(clf, 'logistic_regression_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("Model and vectorizer saved successfully!")